<a href="https://colab.research.google.com/github/ticketguy/littlefig/blob/main/Little_Fig_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🍐 Little Fig v0.6

**CPU-native LLM engine with embedded cognitive memory.**

| Component | What it does |
|---|---|
| FigQuant | Adaptive codebook INT4 (9.7% less MSE than NF4) |
| FigKernel | torch.compile fused ops (2.95× faster RMSNorm) |
| FigPipeline | GPU compute + CPU optimizer states |
| 4 Tiers | LoRA, LISA, MeZO, LOMO (auto-selected) |
| Ember | Cognitive memory tokens embedded in weights |

⭐ [GitHub](https://github.com/ticketguy/littlefig) · [Ember's Diaries](https://github.com/ticketguy/embers-diaries)

## 1. Setup

In [ ]:
# Install Little Fig + dependencies (~2 min)
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121 2>/dev/null || true
!pip install -q git+https://github.com/ticketguy/littlefig.git 2>&1 | tail -3
!pip install -q embers-diaries datasets 2>/dev/null || true

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print("✅ Ready")

## 2. Quick Start — Fine-tune with FigQuant

In [ ]:
from little_fig.engine import FigModel, FigTrainer, FigTrainingConfig, TrainingTier

# Load model — FigQuant quantizes to INT4 + adds LoRA adapters
model = FigModel.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    lora_r=16,
    lora_alpha=32,
    tier=TrainingTier.STREAMING_LORA,
)

model.print_trainable_summary()

In [ ]:
# Configure training with memory optimizations
config = FigTrainingConfig(
    num_epochs=1,
    learning_rate=2e-4,
    max_seq_length=512,
    batch_size=1,
    gradient_accumulation_steps=8,
    activation_checkpointing=True,  # ~70% activation memory savings
    logging_steps=5,
)

trainer = FigTrainer(model, config)
trainer.load_dataset("tatsu-lab/alpaca", max_samples=200)
trainer.train()

# Save adapter
model.save_adapter("./my_adapter")

## 3. Ember Memory — Train with Cognitive Memory

In [ ]:
from little_fig.engine import FigModel, FigTrainer, FigTrainingConfig, MEMORY_TOKENS

# Show the 9 memory tokens
print("Ember memory tokens:")
for name, token in MEMORY_TOKENS.items():
    print(f"  {name:15s} → {token}")

# Load with Ember mode — injects tokens into tokenizer
ember_model = FigModel.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    lora_r=16,
    ember_mode=True,
)

config = FigTrainingConfig(num_epochs=1, max_seq_length=256, logging_steps=5)
trainer = FigTrainer(ember_model, config)

# Generate + load cognitive memory training data
trainer.load_ember_dataset(n_examples=200)
trainer.train()

ember_model.save_adapter("./ember_adapter")
print("\n🔥 Ember memory adapter saved!")

## 4. Benchmarks — FigQuant vs Baseline

In [ ]:
import torch
from little_fig.engine import figquant_quantize, figquant_dequantize, measure_quality
from little_fig.engine import FigRMSNorm
import time

# FigQuant quality
W = torch.randn(2048, 768)
fq = figquant_quantize(W, group_size=128, n_iters=8)
qual = measure_quality(W, fq)
print(f"FigQuant Quality:")
print(f"  Cosine Sim:  {qual['cosine_similarity']:.6f}")
print(f"  MSE:         {qual['mse']:.6f}")
print(f"  SNR:         {qual['snr_db']:.1f} dB")
print(f"  Bits/param:  {qual['bits_per_param']:.2f}")
print(f"  Compression: {qual['compression_ratio']:.1f}×")

# FigKernel RMSNorm speedup
x = torch.randn(4, 256, 2048)
norm = FigRMSNorm(2048)
weight = torch.ones(2048)

# Standard
t0 = time.time()
for _ in range(100): _ = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-6) * weight
std_ms = (time.time() - t0) / 100 * 1000

# FigRMSNorm
t0 = time.time()
for _ in range(100): _ = norm(x)
fig_ms = (time.time() - t0) / 100 * 1000

print(f"\nFigKernel RMSNorm:")
print(f"  Standard: {std_ms:.2f} ms")
print(f"  FigRMSNorm: {fig_ms:.2f} ms")
print(f"  Speedup: {std_ms/fig_ms:.2f}×")

## 5. Launch Studio UI

In [ ]:
import subprocess, time

proc = subprocess.Popen(
    ['python', '-c', 'from little_fig.web.server import run_server; run_server()'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
time.sleep(3)

from google.colab import output
output.serve_kernel_port_as_iframe(8888, height=900, width="100%")
print("🍐 Studio running on port 8888")

## 6. Push to Hub

After training, push your adapter to HuggingFace Hub.

In [ ]:
# Uncomment and run to push your trained adapter
# model.push_to_hub("your-username/my-model", private=True)